# Hotel Bookings: Exploratory Data Analysis

This notebook follows the supplied EDA reference: understand the fields, load the data, inspect quality, engineer useful features, review distributions, and answer focused business questions. All analysis uses only pandas and NumPy, and every visualization uses Plotly.


## Data understanding


| Column | Meaning |
| --- | --- |
| `hotel` | Hotel type: City Hotel or Resort Hotel. |
| `is_canceled` | Target flag: 1 if the booking was canceled, otherwise 0. |
| `lead_time` | Days between booking creation and arrival. |
| `arrival_date_year` | Scheduled arrival year. |
| `arrival_date_month` | Scheduled arrival month name. |
| `arrival_date_week_number` | Week number of the scheduled arrival date. |
| `arrival_date_day_of_month` | Day of month of the scheduled arrival date. |
| `stays_in_weekend_nights` | Number of weekend nights booked. |
| `stays_in_week_nights` | Number of weekday nights booked. |
| `adults` | Number of adults in the booking. |
| `children` | Number of children in the booking. |
| `babies` | Number of babies in the booking. |
| `meal` | Booked meal plan. |
| `country` | Guest country code. |
| `market_segment` | Market segment through which the booking was made. |
| `distribution_channel` | Booking distribution channel. |
| `is_repeated_guest` | Flag showing whether the guest booked before. |
| `previous_cancellations` | Number of earlier bookings canceled by the guest. |
| `previous_bookings_not_canceled` | Number of earlier bookings not canceled by the guest. |
| `reserved_room_type` | Reserved room-type code. |
| `assigned_room_type` | Room-type code ultimately assigned. |
| `booking_changes` | Number of changes made to the booking. |
| `deposit_type` | Deposit category. |
| `agent` | Booking-agent identifier. |
| `company` | Company or entity identifier associated with the booking. |
| `days_in_waiting_list` | Days the booking remained on a waiting list. |
| `customer_type` | Customer category such as Transient or Contract. |
| `adr` | Average daily rate, calculated as lodging revenue divided by occupied room nights. |
| `required_car_parking_spaces` | Number of requested parking spaces. |
| `total_of_special_requests` | Number of special requests made by the guest. |
| `reservation_status` | Final reservation status. This is an outcome field. |
| `reservation_status_date` | Date on which the final reservation status was set. |


## Data loading

The loader checks the current folder, its parent, and the user's Downloads folder. This keeps the notebook portable while still working with the supplied dataset location.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: "{:,.2f}".format(value))
pio.templates.default = "plotly_white"


In [ ]:
DATA_PATH = "hotel_bookings.csv"
raw = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", raw.shape)
raw.head()


## Data exploration


### Data types and non-null counts


In [ ]:
raw.info()


### Numerical summary


In [ ]:
raw.describe().T.round(2)


### Categorical summary


In [ ]:
raw.describe(include=["object"]).T


### Duplicate rows


In [ ]:
duplicate_count = int(raw.duplicated().sum())
duplicate_share = raw.duplicated().mean()
pd.Series({"duplicate_rows": duplicate_count, "duplicate_share": duplicate_share})


The dataset has no reservation identifier, so an exact row match is not definitive proof that two records represent the same booking. Duplicate rows are reported and retained for descriptive booking-volume analysis. The preprocessing notebook removes exact duplicates before the train/test split to reduce evaluation leakage.


### Missing values


In [ ]:
missing = pd.DataFrame(
    {
        "missing_count": raw.isna().sum(),
        "missing_percent": raw.isna().mean().mul(100).round(2),
    }
)
missing.loc[missing["missing_count"] > 0].sort_values("missing_percent", ascending=False)


### Basic validity checks


In [ ]:
validity_checks = pd.Series(
    {
        "rows_with_zero_guests": int(
            ((raw["adults"] + raw["children"].fillna(0) + raw["babies"]) == 0).sum()
        ),
        "rows_with_zero_nights": int(
            ((raw["stays_in_weekend_nights"] + raw["stays_in_week_nights"]) == 0).sum()
        ),
        "rows_with_negative_adr": int((raw["adr"] < 0).sum()),
        "rows_with_missing_arrival_fields": int(
            raw[["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]]
            .isna()
            .any(axis=1)
            .sum()
        ),
    }
)
validity_checks


## Feature engineering and analysis dataset


In [ ]:
df = raw.copy()

# Small, interpretable missing-value treatments for analysis.
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["has_agent"] = df["agent"].notna().astype(int)
df["has_company"] = df["company"].notna().astype(int)

# Derived booking measures.
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["has_children"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)
df["cancellation_label"] = df["is_canceled"].map({0: "Not canceled", 1: "Canceled"})

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
month_number = {month: index for index, month in enumerate(month_order, start=1)}
df["arrival_month_number"] = df["arrival_date_month"].map(month_number)
df["arrival_date"] = pd.to_datetime(
    {
        "year": df["arrival_date_year"],
        "month": df["arrival_month_number"],
        "day": df["arrival_date_day_of_month"],
    },
    errors="coerce",
)
df["lead_time_band"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, np.inf],
    labels=["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181+ days"],
)

# Exclude records that cannot represent a valid occupied booking from rate/ADR analysis.
# Exact duplicate rows are retained for EDA because the data has no booking ID.
analysis_df = df.loc[(df["total_guests"] > 0) & (df["adr"] >= 0)].copy()

print("Rows used for EDA:", len(analysis_df))
print("Excluded zero-guest rows:", int((df["total_guests"] == 0).sum()))
print("Excluded negative-ADR rows:", int((df["adr"] < 0).sum()))
analysis_df[["total_nights", "total_guests", "arrival_date", "lead_time_band"]].head()


## Distribution checks


### Numerical distributions


In [ ]:
numeric_plot_columns = [
    "lead_time",
    "total_nights",
    "total_guests",
    "adr",
    "booking_changes",
    "total_of_special_requests",
]

for column in numeric_plot_columns:
    upper = analysis_df[column].quantile(0.99)
    plot_data = analysis_df.loc[analysis_df[column] <= upper, [column]]
    figure = px.histogram(
        plot_data,
        x=column,
        nbins=40,
        title="{} distribution (up to 99th percentile)".format(
            column.replace("_", " ").title()
        ),
        labels={column: column.replace("_", " ").title()},
    )
    figure.update_layout(yaxis_title="Bookings", showlegend=False)
    figure.show()


### Categorical distributions


In [ ]:
categorical_plot_columns = ["hotel", "customer_type", "market_segment", "deposit_type"]

for column in categorical_plot_columns:
    category_counts = (
        analysis_df[column]
        .value_counts()
        .rename_axis(column)
        .reset_index(name="bookings")
        .sort_values("bookings")
    )
    figure = px.bar(
        category_counts,
        x="bookings",
        y=column,
        orientation="h",
        title="{} bookings".format(column.replace("_", " ").title()),
        labels={column: "", "bookings": "Bookings"},
    )
    figure.show()


## Q1. What does the cancellation distribution look like?


In [ ]:
cancellation_summary = (
    analysis_df["cancellation_label"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="bookings")
)
cancellation_summary["share"] = (
    cancellation_summary["bookings"] / cancellation_summary["bookings"].sum()
)
display(cancellation_summary)

figure = px.bar(
    cancellation_summary,
    x="status",
    y="bookings",
    color="status",
    color_discrete_map={"Not canceled": "#59A14F", "Canceled": "#E15759"},
    title="Booking cancellation distribution",
    labels={"status": "", "bookings": "Bookings"},
)
figure.update_layout(showlegend=False)
figure.show()


## Q2. How does cancellation rate differ by hotel type?


In [ ]:
hotel_cancel = (
    analysis_df.groupby("hotel", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
    .sort_values("cancellation_rate", ascending=False)
)
display(hotel_cancel)

figure = px.bar(
    hotel_cancel,
    x="hotel",
    y="cancellation_rate",
    title="Cancellation rate by hotel type",
    labels={"hotel": "", "cancellation_rate": "Cancellation rate"},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q3. How is lead time related to cancellation?


In [ ]:
lead_cancel = (
    analysis_df.groupby("lead_time_band", observed=False, as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
)
display(lead_cancel)

figure = px.bar(
    lead_cancel,
    x="lead_time_band",
    y="cancellation_rate",
    title="Cancellation rate by lead-time band",
    labels={"lead_time_band": "Lead time", "cancellation_rate": "Cancellation rate"},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q4. How do monthly cancellation rates vary by hotel?


In [ ]:
monthly_cancel = (
    analysis_df.groupby(["arrival_date_month", "hotel"], as_index=False)["is_canceled"]
    .mean()
    .rename(columns={"is_canceled": "cancellation_rate"})
)
monthly_cancel["arrival_date_month"] = pd.Categorical(
    monthly_cancel["arrival_date_month"], categories=month_order, ordered=True
)
monthly_cancel = monthly_cancel.sort_values("arrival_date_month")

figure = px.line(
    monthly_cancel,
    x="arrival_date_month",
    y="cancellation_rate",
    color="hotel",
    markers=True,
    title="Monthly cancellation rate by hotel",
    labels={
        "arrival_date_month": "Arrival month",
        "cancellation_rate": "Cancellation rate",
        "hotel": "Hotel",
    },
    category_orders={"arrival_date_month": month_order},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q5. Which market segments have the highest cancellation rates?


In [ ]:
segment_cancel = (
    analysis_df.groupby("market_segment", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
    .sort_values("cancellation_rate")
)
display(segment_cancel.sort_values("cancellation_rate", ascending=False))

figure = px.bar(
    segment_cancel,
    x="cancellation_rate",
    y="market_segment",
    orientation="h",
    title="Cancellation rate by market segment",
    labels={"market_segment": "", "cancellation_rate": "Cancellation rate"},
)
figure.update_xaxes(tickformat=".0%")
figure.show()


## Q6. How does average daily rate vary by hotel and cancellation status?


In [ ]:
adr_cap = analysis_df["adr"].quantile(0.99)
adr_plot = analysis_df.loc[analysis_df["adr"] <= adr_cap]

figure = px.box(
    adr_plot,
    x="hotel",
    y="adr",
    color="cancellation_label",
    points=False,
    title="Average daily rate by hotel and cancellation status",
    labels={"hotel": "", "adr": "ADR", "cancellation_label": "Status"},
)
figure.show()


## Q7. How does deposit type relate to cancellation?


In [ ]:
deposit_cancel = (
    analysis_df.groupby("deposit_type", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
    .sort_values("cancellation_rate", ascending=False)
)
display(deposit_cancel)

figure = px.bar(
    deposit_cancel,
    x="deposit_type",
    y="cancellation_rate",
    title="Cancellation rate by deposit type",
    labels={"deposit_type": "", "cancellation_rate": "Cancellation rate"},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q8. Do special requests correspond to lower cancellation rates?


In [ ]:
request_cancel = (
    analysis_df.groupby("total_of_special_requests", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
)
display(request_cancel)

figure = px.line(
    request_cancel,
    x="total_of_special_requests",
    y="cancellation_rate",
    markers=True,
    title="Cancellation rate by number of special requests",
    labels={
        "total_of_special_requests": "Special requests",
        "cancellation_rate": "Cancellation rate",
    },
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q9. Do repeat guests cancel less often?


In [ ]:
repeat_cancel = (
    analysis_df.assign(
        guest_history=analysis_df["is_repeated_guest"].map({0: "New guest", 1: "Repeat guest"})
    )
    .groupby("guest_history", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
)
display(repeat_cancel)

figure = px.bar(
    repeat_cancel,
    x="guest_history",
    y="cancellation_rate",
    title="Cancellation rate by guest history",
    labels={"guest_history": "", "cancellation_rate": "Cancellation rate"},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q10. Which countries contribute the most bookings, and how do their cancellation rates compare?


In [ ]:
top_countries = analysis_df["country"].value_counts().head(10).index
country_summary = (
    analysis_df.loc[analysis_df["country"].isin(top_countries)]
    .groupby("country", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
    .sort_values("bookings", ascending=False)
)
display(country_summary)

figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Bookings from the ten largest source countries", "Cancellation rate"),
    horizontal_spacing=0.15,
)
figure.add_trace(
    go.Bar(x=country_summary["country"], y=country_summary["bookings"], name="Bookings"),
    row=1,
    col=1,
)
figure.add_trace(
    go.Bar(
        x=country_summary["country"],
        y=country_summary["cancellation_rate"],
        name="Cancellation rate",
    ),
    row=1,
    col=2,
)
figure.update_yaxes(title_text="Bookings", row=1, col=1)
figure.update_yaxes(title_text="Cancellation rate", tickformat=".0%", row=1, col=2)
figure.update_layout(title="Top source countries", showlegend=False)
figure.show()


## Q11. Are room-type changes associated with cancellation?


In [ ]:
room_change_cancel = (
    analysis_df.assign(
        room_assignment=analysis_df["room_changed"].map(
            {0: "Reserved type assigned", 1: "Different type assigned"}
        )
    )
    .groupby("room_assignment", as_index=False)
    .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
)
display(room_change_cancel)

figure = px.bar(
    room_change_cancel,
    x="room_assignment",
    y="cancellation_rate",
    title="Cancellation rate by room assignment outcome",
    labels={"room_assignment": "", "cancellation_rate": "Cancellation rate"},
)
figure.update_yaxes(tickformat=".0%")
figure.show()


## Q12. Which numerical variables are most associated with cancellation?


In [ ]:
correlation_columns = [
    "is_canceled",
    "lead_time",
    "total_nights",
    "total_guests",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
]
correlation_matrix = analysis_df[correlation_columns].corr()

figure = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.index,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Correlation"),
    )
)
figure.update_layout(
    title="Correlation matrix for cancellation-related numerical variables",
    width=950,
    height=750,
)
figure.show()

correlation_matrix["is_canceled"].drop("is_canceled").sort_values(
    key=np.abs, ascending=False
)


## Final data-quality summary


In [ ]:
final_summary = pd.Series(
    {
        "source_rows": len(raw),
        "eda_rows": len(analysis_df),
        "source_columns": raw.shape[1],
        "engineered_columns": analysis_df.shape[1] - raw.shape[1],
        "cancellation_rate": analysis_df["is_canceled"].mean(),
        "date_parse_failures": int(analysis_df["arrival_date"].isna().sum()),
        "remaining_missing_children": int(analysis_df["children"].isna().sum()),
        "remaining_missing_country": int(analysis_df["country"].isna().sum()),
    }
)
final_summary


## Optional export

Set `EXPORT_CLEANED` to `True` to save the analysis-ready data. The default is `False` so running the notebook produces no unrequested files.


In [ ]:
EXPORT_CLEANED = False

if EXPORT_CLEANED:
    export_path = "hotel_bookings_cleaned.csv"
    analysis_df.to_csv(export_path, index=False)
    print("Saved:", export_path)
else:
    print("Export skipped. Set EXPORT_CLEANED = True to save the cleaned dataset.")


In [ ]:
%%writefile hotel_bookings_app.py
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st


st.set_page_config(page_title="Hotel Booking Analysis", page_icon="🏨", layout="wide")


@st.cache_resource(show_spinner=False)
def load_data(path="hotel_bookings.csv"):
    """Read the CSV in bounded chunks and retain only aggregates plus a preview."""
    use_columns = [
        "hotel",
        "is_canceled",
        "lead_time",
        "arrival_date_month",
        "adults",
        "children",
        "babies",
        "market_segment",
        "adr",
    ]
    column_types = {
        "hotel": "category",
        "is_canceled": "int8",
        "lead_time": "int16",
        "arrival_date_month": "category",
        "adults": "int16",
        "children": "float32",
        "babies": "int16",
        "market_segment": "category",
        "adr": "float32",
    }
    lead_labels = ["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181+ days"]
    group_columns = [
        "hotel",
        "arrival_date_month",
        "market_segment",
        "is_canceled",
        "lead_time_band",
    ]

    aggregate_parts = []
    preview_parts = []
    preview_rows_remaining = 5000

    reader = pd.read_csv(
        path,
        usecols=use_columns,
        dtype=column_types,
        chunksize=10000,
    )

    for chunk in reader:
        chunk["children"] = chunk["children"].fillna(0).astype("int16")
        chunk["total_guests"] = (
            chunk["adults"] + chunk["children"] + chunk["babies"]
        ).astype("int16")
        chunk["lead_time_band"] = pd.cut(
            chunk["lead_time"],
            bins=[-1, 7, 30, 90, 180, np.inf],
            labels=lead_labels,
        )
        chunk = chunk.loc[(chunk["total_guests"] > 0) & (chunk["adr"] >= 0)]

        grouped = (
            chunk.groupby(group_columns, observed=True, as_index=False)
            .agg(
                bookings=("is_canceled", "size"),
                adr_sum=("adr", "sum"),
                lead_time_sum=("lead_time", "sum"),
            )
        )
        aggregate_parts.append(grouped)

        if preview_rows_remaining > 0:
            preview = chunk.head(preview_rows_remaining).copy()
            preview["cancellation_label"] = preview["is_canceled"].map(
                {0: "Not canceled", 1: "Canceled"}
            )
            preview_parts.append(preview)
            preview_rows_remaining -= len(preview)

    summary = pd.concat(aggregate_parts, ignore_index=True)
    summary = (
        summary.groupby(group_columns, observed=True, as_index=False)[
            ["bookings", "adr_sum", "lead_time_sum"]
        ]
        .sum()
    )
    summary["cancellation_label"] = summary["is_canceled"].map(
        {0: "Not canceled", 1: "Canceled"}
    ).astype("category")

    preview = pd.concat(preview_parts, ignore_index=True)
    return summary, preview


summary, preview = load_data()

st.title("Hotel Booking Analysis")
st.caption(
    "Interactive analysis generated from chunked CSV aggregates. "
    "The app never holds the complete source table in memory."
)

st.sidebar.header("Filters")

hotel_options = sorted(summary["hotel"].dropna().astype(str).unique().tolist())
selected_hotels = st.sidebar.multiselect("Hotel", hotel_options, default=hotel_options)

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
months_present = set(summary["arrival_date_month"].dropna().astype(str))
available_months = [month for month in month_order if month in months_present]
selected_months = st.sidebar.multiselect(
    "Arrival month", available_months, default=available_months
)

segment_options = sorted(summary["market_segment"].dropna().astype(str).unique().tolist())
selected_segments = st.sidebar.multiselect(
    "Market segment", segment_options, default=segment_options
)

status_options = ["Not canceled", "Canceled"]
selected_statuses = st.sidebar.multiselect(
    "Booking status", status_options, default=status_options
)

# The aggregate table is small. Filtering it is inexpensive and bounded.
filtered = summary
if set(selected_hotels) != set(hotel_options):
    filtered = filtered.loc[filtered["hotel"].astype(str).isin(selected_hotels)]
if set(selected_months) != set(available_months):
    filtered = filtered.loc[filtered["arrival_date_month"].astype(str).isin(selected_months)]
if set(selected_segments) != set(segment_options):
    filtered = filtered.loc[filtered["market_segment"].astype(str).isin(selected_segments)]
if set(selected_statuses) != set(status_options):
    filtered = filtered.loc[filtered["cancellation_label"].astype(str).isin(selected_statuses)]

if filtered.empty:
    st.warning("No bookings match the selected filters.")
    st.stop()

filtered = filtered.assign(
    canceled_bookings=filtered["bookings"] * filtered["is_canceled"]
)
booking_count = int(filtered["bookings"].sum())
canceled_count = int(filtered["canceled_bookings"].sum())
average_adr = filtered["adr_sum"].sum() / booking_count
average_lead_time = filtered["lead_time_sum"].sum() / booking_count

metric_1, metric_2, metric_3, metric_4 = st.columns(4)
metric_1.metric("Bookings", "{:,.0f}".format(booking_count))
metric_2.metric("Cancellation rate", "{:.1%}".format(canceled_count / booking_count))
metric_3.metric("Average ADR", "{:,.2f}".format(average_adr))
metric_4.metric("Average lead time", "{:,.1f} days".format(average_lead_time))

left, right = st.columns(2)

with left:
    status_summary = (
        filtered.groupby("cancellation_label", observed=True, as_index=False)["bookings"]
        .sum()
    )
    figure = px.bar(
        status_summary,
        x="cancellation_label",
        y="bookings",
        color="cancellation_label",
        color_discrete_map={"Not canceled": "#59A14F", "Canceled": "#E15759"},
        title="Booking status",
        labels={"cancellation_label": "", "bookings": "Bookings"},
    )
    figure.update_layout(showlegend=False)
    st.plotly_chart(figure, use_container_width=True)

with right:
    hotel_summary = (
        filtered.groupby("hotel", observed=True, as_index=False)[
            ["bookings", "canceled_bookings"]
        ]
        .sum()
    )
    hotel_summary["cancellation_rate"] = (
        hotel_summary["canceled_bookings"] / hotel_summary["bookings"]
    )
    figure = px.bar(
        hotel_summary,
        x="hotel",
        y="cancellation_rate",
        color="hotel",
        title="Cancellation rate by hotel",
        labels={"hotel": "", "cancellation_rate": "Cancellation rate"},
    )
    figure.update_yaxes(tickformat=".0%")
    figure.update_layout(showlegend=False)
    st.plotly_chart(figure, use_container_width=True)

monthly_summary = (
    filtered.groupby(["arrival_date_month", "hotel"], observed=True, as_index=False)[
        ["bookings", "canceled_bookings"]
    ]
    .sum()
)
monthly_summary["cancellation_rate"] = (
    monthly_summary["canceled_bookings"] / monthly_summary["bookings"]
)
monthly_summary["arrival_date_month"] = pd.Categorical(
    monthly_summary["arrival_date_month"], categories=month_order, ordered=True
)
monthly_summary = monthly_summary.sort_values("arrival_date_month")

figure = px.line(
    monthly_summary,
    x="arrival_date_month",
    y="cancellation_rate",
    color="hotel",
    markers=True,
    title="Monthly cancellation rate",
    labels={
        "arrival_date_month": "Arrival month",
        "cancellation_rate": "Cancellation rate",
        "hotel": "Hotel",
    },
    category_orders={"arrival_date_month": month_order},
)
figure.update_yaxes(tickformat=".0%")
st.plotly_chart(figure, use_container_width=True)

left, right = st.columns(2)

with left:
    lead_summary = (
        filtered.groupby("lead_time_band", observed=True, as_index=False)[
            ["bookings", "canceled_bookings"]
        ]
        .sum()
    )
    lead_summary["cancellation_rate"] = (
        lead_summary["canceled_bookings"] / lead_summary["bookings"]
    )
    figure = px.bar(
        lead_summary,
        x="lead_time_band",
        y="cancellation_rate",
        title="Cancellation rate by lead time",
        labels={"lead_time_band": "Lead time", "cancellation_rate": "Cancellation rate"},
    )
    figure.update_yaxes(tickformat=".0%")
    st.plotly_chart(figure, use_container_width=True)

with right:
    segment_summary = (
        filtered.groupby("market_segment", observed=True, as_index=False)[
            ["bookings", "canceled_bookings"]
        ]
        .sum()
    )
    segment_summary["cancellation_rate"] = (
        segment_summary["canceled_bookings"] / segment_summary["bookings"]
    )
    segment_summary = segment_summary.sort_values("cancellation_rate")
    figure = px.bar(
        segment_summary,
        x="cancellation_rate",
        y="market_segment",
        orientation="h",
        title="Cancellation rate by market segment",
        labels={"market_segment": "", "cancellation_rate": "Cancellation rate"},
    )
    figure.update_xaxes(tickformat=".0%")
    st.plotly_chart(figure, use_container_width=True)

adr_summary = (
    filtered.groupby(["hotel", "cancellation_label"], observed=True, as_index=False)[
        ["bookings", "adr_sum"]
    ]
    .sum()
)
adr_summary["average_adr"] = adr_summary["adr_sum"] / adr_summary["bookings"]
figure = px.bar(
    adr_summary,
    x="hotel",
    y="average_adr",
    color="cancellation_label",
    barmode="group",
    title="Average daily rate by hotel and booking status",
    labels={"hotel": "", "average_adr": "Average ADR", "cancellation_label": "Status"},
)
st.plotly_chart(figure, use_container_width=True)

st.subheader("Booking sample")
preview_filtered = preview
if set(selected_hotels) != set(hotel_options):
    preview_filtered = preview_filtered.loc[
        preview_filtered["hotel"].astype(str).isin(selected_hotels)
    ]
if set(selected_months) != set(available_months):
    preview_filtered = preview_filtered.loc[
        preview_filtered["arrival_date_month"].astype(str).isin(selected_months)
    ]
if set(selected_segments) != set(segment_options):
    preview_filtered = preview_filtered.loc[
        preview_filtered["market_segment"].astype(str).isin(selected_segments)
    ]
if set(selected_statuses) != set(status_options):
    preview_filtered = preview_filtered.loc[
        preview_filtered["cancellation_label"].isin(selected_statuses)
    ]

preview_columns = [
    "hotel",
    "is_canceled",
    "lead_time",
    "arrival_date_month",
    "market_segment",
    "adr",
    "total_guests",
]
st.dataframe(preview_filtered.loc[:, preview_columns], use_container_width=True)
st.caption("The table is a bounded sample of up to 5,000 source rows.")

# Run from a terminal in the project folder:
# streamlit run hotel_bookings_app.py
